# D450 — Spark Structured Streaming Introduction

### From a fixed dataset to data that keeps arriving

Streams → batch versus streaming → incremental processing → Spark's table model → micro-batches → queries and triggers → time and recovery.

Code snippets use **PySpark 3.5.7** to connect each concept to a business situation. Each snippet is read independently: names such as `transactions`, `alerts` and `file_events` stand for the data described on that slide. No setup or execution sequence is required.

Reference: [Apache Spark 3.5.7 Structured Streaming Programming Guide](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html). Diagrams are the original images from that guide, stored locally for offline reading.

## 1. What is a stream?

A **stream** is a sequence of events arriving over time, potentially without an end.

- A payment happens, a customer clicks, a sensor reports a temperature.
- Each occurrence becomes an **event record** with fields describing what happened.
- New records keep arriving while earlier records are being processed.

```text
10:00:01  T101  account=A7  amount=500
10:00:02  T102  account=A2  amount=12000
10:00:04  T103  account=A7  amount=800
... more transactions arrive ...
```

There is usually no final row that tells us “the entire stream is complete.”

## 2. Follow one event through a pipeline

```text
Payment application → Kafka topic → Spark query → Alert records → Review system
      producer          source       processor       sink        consumer
```

| Term | Meaning |
|---|---|
| Producer | Creates an event |
| Source | Makes events available for processing |
| Processor | Filters, enriches or aggregates events |
| Sink | Receives computed results |
| Offset / progress | Identifies how far processing has reached |

A stream need not arrive through Kafka. A directory receiving new, complete files can also be a streaming source.

## 3. Bounded and unbounded data

**Bounded:** “Read all payments in yesterday's file.” The input selected for this run has an end.

**Unbounded:** “Keep reading payments as they arrive.” The computation cannot wait for all future input.

The same business data can be viewed either way:

- Historical payments → monthly reconciliation.
- Arriving payments → timely suspicious-activity detection.

Streaming therefore needs a rule for **when to produce an intermediate result**, even though more input may arrive.

## 4. Batch processing versus stream processing

| Aspect | Batch processing | Stream processing |
|---|---|---|
| Input for a computation | A bounded selection | Ongoing arrivals |
| Execution | A job handles the selected input | A query repeatedly or continuously handles new input |
| Result availability | After the relevant batch finishes | As processing advances |
| Common priority | Throughput and cost per run | Freshness and sustained throughput |
| Main questions | Did the job finish? Can it be rerun? | Is the query keeping up? Can it recover? |
| Example | Daily revenue report | Payment alerts and live metrics |

Neither is automatically faster or cheaper. Choose according to how quickly the result must influence a decision.

## 5. When is streaming useful?

| Situation | Useful result | Why freshness matters |
|---|---|---|
| Suspicious payments | Flag an unusual transaction pattern | Investigate before more activity occurs |
| Application errors | Updated error-rate alert | Shorten incident response time |
| Factory sensors | Overheating warning | React while equipment is operating |
| Online shopping | Current conversion metrics | Observe a campaign while it runs |
| Vehicle locations | Updated delivery status | Support current routing decisions |

Batch fits reports and backfills when waiting is acceptable. Streaming fits decisions whose value decreases with delay.

## 6. Real-time means a measurable deadline

“Real-time” should become a requirement such as **“an alert is visible within 10 seconds.”**

```text
Total delay ≈ source delay + waiting for execution + processing + sink delivery
```

- **Latency:** how long an event takes to influence an available result.
- **Throughput:** how many events can be processed per second.
- **Backlog / lag:** input that has arrived but is not yet processed.

If 10,000 events arrive each second but the pipeline processes 8,000, backlog grows. A shorter trigger interval alone does not add processing capacity.

## 7. Fraud detection: from one event to a pattern

| Rule | Example | Information required |
|---|---|---|
| Large payment | Amount ≥ 10,000 | Current event only |
| High activity | At least 3 payments in a 5-minute window | Per-account counts across events |
| Risk context | Large payment from a high-risk account | Event plus reference data |

The first rule is **stateless**. The second is **stateful** because a later event needs information retained from earlier events.

These rules create candidates for review. A full fraud system also needs validated rules, false-positive handling and reliable alert delivery.

## 8. What is incremental data processing?

Process the **new or changed portion** since the previous successful run and update the result.

| Run | Newly arrived amounts | Previous total | Updated total |
|---|---|---|---|
| 1 | 100, 200 | 0 | 300 |
| 2 | 50 | 300 | 350 |
| 3 | 80, 20 | 350 | 450 |

```text
updated result = update(previous result, new input)
```

Run 3 can add 100 to the saved total instead of summing all five payments again. The saved total is **state**; the saved input position is **progress**.

## 9. Incremental does not mean always running

| Approach | Example |
|---|---|
| Full batch recomputation | Recalculate revenue from every historical payment |
| Scheduled incremental job | Every hour, add newly arrived payments |
| Ongoing streaming query | Process arrivals throughout the day |

All incremental approaches need to answer:

1. What input is new?
2. Which result/state must change?
3. How do retries avoid losing or duplicating effects?

For updates and deletes, define change semantics explicitly. Appending a correction event does not automatically replace an earlier output row. For an average, retain **sum and count**, not just the previous average.

## 10. Introducing Spark Structured Streaming

Structured Streaming applies familiar DataFrame and SQL expressions to arriving data. Spark plans how supported computations can advance incrementally.

```text
readStream → select / filter / groupBy → writeStream → start()
```

- Reuse Spark SQL expressions, schemas and distributed execution concepts.
- Add source progress, output behavior, scheduling and query lifecycle.
- Understand which operations need retained state.

**Terminology:** this lesson uses Structured Streaming. The older Spark Streaming API uses **DStreams** and `StreamingContext`; it is a different API.

## 11. Diagram: a stream becomes an input table

<img src="assets/structured-streaming-stream-as-a-table.png" alt="Spark diagram: arriving stream records become rows appended to an unbounded input table" style="background-color:white;max-width:100%;max-height:440px;object-fit:contain;" />

**Read the arrows:** each arriving record becomes another row. The schema can stay fixed while the number of rows keeps growing.

This is a **logical table model**. It does not mean Spark must load and retain the entire stream history in memory.

Source: [Spark guide — Basic Concepts](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#basic-concepts).

## 12. Diagram: input table, result table and output

<img src="assets/structured-streaming-model.png" alt="Spark diagram: input grows over triggers, a query updates the result table, and output is sent to a sink" style="background-color:white;max-width:100%;max-height:420px;object-fit:contain;" />

Read each vertical column: **input so far → query result → output**. Then move left to right to see processing advance.

The **result table** represents the answer. **Output** is the portion of that answer sent to the sink, selected by the output mode.

Source: [Spark guide — Programming Model](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#programming-model).

## 13. Diagram: a running word count

<img src="assets/structured-streaming-example-model.png" alt="Spark word-count diagram showing cat, dog and owl counts updated as new lines arrive" style="background-color:white;max-width:100%;max-height:420px;object-fit:contain;" />

Follow `cat`: its count changes from **1 → 2 → 2**. Follow `dog`: **3 → 3 → 4**.

The query combines new word occurrences with saved counts. The growing input table explains the answer; rereading all old input is unnecessary for this aggregation.

Source: [Spark guide — Basic Concepts](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#basic-concepts).

## 14. Micro-batches: small batches inside one query

A **micro-batch** processes an admitted portion of input. An ongoing query executes a sequence of these batches.

```text
New input:       a b c          d e           f g
Processing:     [batch 0]      [batch 1]     [batch 2] ...
State:          update        update       update
```

For each batch, Spark determines input progress, computes results, writes output and records successful progress for recovery.

**One query ≠ one batch ≠ one Spark job.** A micro-batch can launch multiple jobs and tasks. Micro-batch size depends on arrivals, backlog and source limits.

## 15. Example: screen each arriving payment

**Context:** a payments application supplies a streaming DataFrame named `transactions`, with transaction ID, account, event time and amount in a common currency.

**Question:** which payments meet a high-value review threshold?

```python
alerts = transactions.filter("amount >= 10000")
```

| Arriving payment | Amount | Result |
|---|---|---|
| T101 | 500 | No alert |
| T102 | 12000 | Include in alert results |
| T103 | 800 | No alert |

**Interpretation:** apply the familiar DataFrame filter to each new payment. This rule needs no history from previous payments, so it is stateless.

## 16. Lazy evaluation: definition is not execution

**Context:** `alerts` describes a high-value payment filter over a streaming source. Defining that filter builds a plan; it does **not start ongoing processing**.

```python
print(alerts.isStreaming)   # True: the plan contains a streaming source
alerts.printSchema()       # Inspect schema
alerts.explain()           # Inspect plan
```

| Batch DataFrame | Streaming DataFrame |
|---|---|
| Transformations build a plan | Transformations build a plan |
| `show()`, `collect()` or a write can execute bounded work | `writeStream.start()` begins query execution |

Do not call `alerts.show()` or `alerts.collect()` directly: this streaming computation needs a streaming execution path. `isStreaming=True` does not mean a query is active.

## 17. Start the query and keep its handle

**Context:** `alerts` is a streaming payment filter. We want to see newly flagged payments every five seconds. Here the console represents an inspection sink and `checkpoint_path` represents this query's recovery location.

```python
query = (alerts.writeStream
         .format("console")
         .outputMode("append")
         .option("checkpointLocation", checkpoint_path)
         .trigger(processingTime="5 seconds")
         .queryName("large_payments")
         .start())
```

| Part | Question answered |
|---|---|
| `alerts` | What result should be computed? |
| `format("console")` | Where should output go? |
| `outputMode("append")` | Which result rows should be emitted? |
| `trigger(...)` | When/how should execution proceed? |
| Checkpoint | Where should recovery progress be kept? |

`.start()` returns a **StreamingQuery**, an execution handle. It runs in the background; console output may appear in driver logs.

## 18. Output mode: what is written?

| Mode | Meaning | Example |
|---|---|---|
| Append | Emit newly added result rows | Individual high-value payments |
| Update | Emit result rows changed by this trigger | Changed running account counts |
| Complete | Emit the whole current result table | Small running word-count demonstration |

Suppose counts were `A=2, B=1`, and only A receives another event:

- **Update:** emit `A=3`.
- **Complete:** emit `A=3, B=1`.
- **Append:** cannot emit this unfinished running count as a new final row.

Query and sink support determine valid modes. Console/memory sinks are for inspection, not durable delivery. The file sink supports Append.

## 19. Observe a query

**Context:** `query` is the handle of an active payment-screening query. An empty alert output could mean no suspicious payments, no incoming data, or a processing problem. Inspect activity and progress to distinguish them.

```python
print(query.name)          # User-supplied name
print(query.id)            # Preserved when restarting from its checkpoint
print(query.runId)         # New for each start
print(query.isActive)      # Is this execution running?
print(query.status)        # Current activity
print(query.lastProgress)  # Last completed progress report; initially may be None
```

In `lastProgress`, inspect `numInputRows`, `durationMs`, `inputRowsPerSecond` and `processedRowsPerSecond`.

Watch trends together with source lag. A batch can receive input but emit no alerts because no rows match the filter.

## 20. Stop a query; understand waiting

**Context:** `query` is a running payment-screening query. We want to wait briefly for termination, then explicitly stop it for maintenance.

```python
finished = query.awaitTermination(10)  # Wait up to 10 seconds
# False on timeout: the query can still be running.

query.stop()                          # Stop this query
print(query.isActive)                 # False after stop completes
print(query.exception())              # Recorded failure, if one occurred
```

- `awaitTermination()` without a timeout waits until termination; failures raise an exception.
- Waiting does not request a stop. A notebook cell finishing does not stop a background query.
- `query.stop()` keeps the SparkSession available. It does not promise to drain all source backlog.
- Restart by calling `.start()` again with compatible settings and the saved checkpoint; there is no `query.start()` restart method.

## 21. A trigger decides when/how execution proceeds

A **trigger** controls query scheduling or selects the continuous engine. Choose one trigger configuration per query.

```text
Trigger interval: every 5 seconds → when to process
Event-time window: 5 minutes     → which events to group
```

Those settings are independent. A five-minute aggregate can be refreshed every five seconds.

Each trigger slide describes an independent scheduling decision. `alerts` means a stream of flagged payments; `file_events` means rows from newly arriving files. A writer describes configuration; `.start()` is the boundary that begins execution.

Reference: [Spark guide — Triggers](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#triggers).

## 22. Default trigger — process as soon as possible

```python
writer = alerts.writeStream.format("console").outputMode("append")
# No .trigger(...) setting; writer.start() starts the query.
```

**Behavior:** micro-batches are scheduled as promptly as execution permits, without a user-defined fixed interval.

**Use when:** an ongoing query should handle input promptly and the overhead of frequent batches is acceptable.

**Tradeoff:** many small batches can create scheduling overhead. Default does not mean record-at-a-time execution.

## 23. ProcessingTime trigger — a fixed interval

```python
writer = (alerts.writeStream.format("console")
          .trigger(processingTime="5 seconds"))
```

**Behavior:** schedule micro-batches at the requested cadence.

- Batch takes 2 seconds → wait until the next scheduled boundary.
- Batch takes 8 seconds → begin the next batch after completion, without waiting for another boundary.
- Spark does not overlap this query's micro-batches to catch up.

**Use when:** a dashboard or alert pipeline can refresh every few seconds.

**Tradeoff:** a longer interval allows more input to accumulate. A 5-second trigger does not guarantee 5-second end-to-end latency.

## 24. AvailableNow trigger — catch up and finish

```python
# Context: file_events represents newly landed payment files.
writer = (file_events.writeStream.format("parquet")
          .trigger(availableNow=True))
```

**Behavior:** process the input available at the start of the run, potentially across multiple micro-batches, then terminate. A later run handles later arrivals.

**Use when:** hourly incremental ingestion or a scheduled catch-up job should release compute after finishing.

**Tradeoff:** freshness follows the external schedule. Use source limits such as `maxFilesPerTrigger` to manage batch size; confirm source support for AvailableNow.

Reuse the pipeline's checkpoint on the next run. This combines streaming progress tracking with bounded execution.

## 25. Once trigger — legacy one-batch execution

```python
writer = (file_events.writeStream.format("parquet")
          .trigger(once=True))
```

**Behavior:** process available input in one micro-batch and stop.

**Use when:** reading or maintaining older implementations. **Deprecated in Spark 3.5.7**; prefer AvailableNow for new catch-up pipelines.

**Tradeoff:** a large backlog becomes one large batch. AvailableNow can divide that work into several batches and advance watermark processing between them.

Once means “one micro-batch,” not “one record” or “one file.”

## 26. Continuous trigger — a different execution engine

**Context:** `kafka_events` represents events read from Kafka. The requirement is to forward selected fields with very low latency; no aggregation is needed.

```python
writer = (kafka_events.select("key", "value").writeStream
          .format("kafka")
          .trigger(continuous="1 second"))
```

This excerpt shows the execution choice; connection and destination configuration are omitted.

**Behavior:** use continuous processing. The interval controls **checkpointing**, not micro-batch scheduling.

**Use when:** evaluating very low latency for a supported pipeline. Spark 3.5.7 labels this mode **experimental**.

**Tradeoffs:** at-least-once semantics; restricted sources, sinks and operations. Simple projections/filters are supported; aggregations and `foreachBatch` are not. Task failures require restarting the query.

Do not switch a windowed fraud aggregation to continuous mode merely by changing its trigger.

## 27. Choose the trigger from the requirement

| Requirement | Starting choice | Reason |
|---|---|---|
| Ongoing, prompt micro-batch execution | Default | No explicit interval to wait for |
| Refresh every 10 seconds | ProcessingTime | Set an execution cadence |
| Hourly incremental file ingestion | AvailableNow | Process backlog and terminate |
| Understand an older catch-up job | Once | Recognize legacy behavior |
| Specialized low-latency evaluation | Continuous | Check support and delivery semantics |

For ordinary ongoing workloads, begin with micro-batches. Measure actual latency, sustained capacity and sink performance before tuning the interval.

## 28. Incremental file ingestion: hourly payment imports

**Context:** a partner delivers complete payment files throughout the day. An hourly job should process the files it has not handled before, then finish.

**Reader excerpt:** `incoming_path` is the landing location and `payment_schema` describes its records.

```python
file_events = (spark.readStream.schema(payment_schema)
    .option("maxFilesPerTrigger", 1)
    .json(incoming_path))
```

**Writer excerpt:** retain the same checkpoint for this pipeline's scheduled runs.

```python
writer = (file_events.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True))
```

**Interpretation:** admit one file per micro-batch, finish the available backlog, and remember progress for the next run. Sink configuration is omitted to focus on incremental behavior. Publish complete files atomically; do not overwrite ingested files.

## 29. Trace two catch-up runs

| Time | Files available | Work with the saved checkpoint |
|---|---|---|
| First start | A.json, B.json | Process A and B, in separate batches under the one-file limit |
| First completion | A and B are recorded as processed | Query stops |
| Later | C.json arrives | No running query yet |
| Next scheduled start | A, B, C | Process C; prior files remain tracked |
| Another start, no new files | A, B, C | No additional business rows to ingest |

This is **incremental processing**: keep progress across runs, even though execution is not always active.

File progress does not deduplicate business events repeated in a new file. That requires an event key and a separate policy.

## 30. Event time versus processing time

| Timestamp | Meaning | Example |
|---|---|---|
| Event time | When the payment happened | 10:01 |
| Processing time | When Spark processes the record | 10:04 |

A device may lose connectivity and send buffered events later. Arrival order is therefore not necessarily event-time order.

For “payments that happened between 10:00 and 10:05,” group by **event time**. Otherwise an old payment could be counted in a later arrival period.

Earlier results may need to change when delayed events arrive.

## 31. Diagram: grouping events into windows

<img src="assets/structured-streaming-window.png" alt="Spark diagram: events grouped into overlapping 10-minute windows starting every 5 minutes" style="background-color:white;max-width:100%;max-height:400px;object-fit:contain;" />

The guide uses **10-minute windows starting every 5 minutes**. The event at 12:07 belongs to both `[12:00, 12:10)` and `[12:05, 12:15)`.

Window starts are inclusive and ends are exclusive. Overlap intentionally lets one event contribute to multiple groups; the window slide is not the trigger interval.

Source: [Spark guide — Window Operations](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#window-operations-on-event-time).

## 32. Diagram: three window shapes

<img src="assets/structured-streaming-time-window-types.jpg" alt="Spark diagram comparing tumbling, sliding and session windows" style="background-color:white;max-width:100%;max-height:360px;object-fit:contain;" />

| Window | Grouping rule | Example |
|---|---|---|
| Tumbling | Fixed, non-overlapping periods | Revenue per 5-minute period |
| Sliding | Fixed width with overlapping starts | Activity in 5-minute windows starting every minute |
| Session | Activity groups separated by an inactivity gap | A browsing visit |

Source: [Spark guide — Types of time windows](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#types-of-time-windows).

## 33. Diagram: late events can revise earlier results

<img src="assets/structured-streaming-late-data.png" alt="Spark diagram showing a late event updating counts for earlier event-time windows" style="background-color:white;max-width:100%;max-height:410px;object-fit:contain;" />

Read an event's timestamp separately from where it arrives on the processing timeline. A delayed event belongs in its event-time window, not automatically the newest window.

Keeping every old window forever would grow state without a bound. We need a policy for how long old results remain open to revision.

Source: [Spark guide — Handling Late Data and Watermarking](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#handling-late-data-and-watermarking).

## 34. Watermarks: progress in event time

For a single event-time input, think of the watermark as progress based on:

```text
maximum event time observed − configured lateness allowance
```

Example: latest observed event time 12:21, allowance 10 minutes → watermark progresses toward 12:11 as batches advance.

- Eligible old window state can be removed after the watermark passes the window end.
- Events less delayed than the allowance are protected from being dropped as too late.
- Older events may or may not be processed; the guarantee is one-sided.
- Watermarks advance with observed event time and execution, not simply with the wall clock.

No newer event timestamps can mean no watermark progress, even while time passes.

## 35. Diagram: watermark with Update output

<img src="assets/structured-streaming-watermark-update-mode.png" alt="Spark watermark diagram: revised rows are emitted in Update mode and expired window state is removed" style="background-color:white;max-width:100%;max-height:440px;object-fit:contain;" />

Follow **blue** maximum event time → **orange** watermark → **purple** changed output rows. In the guide's example, the late 12:04 event arrives after its old window has expired and is ignored.

Update emits changing counts before finalization. Consumers must handle multiple revisions of the same window key.

Source: [Spark guide — Watermarking](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#handling-late-data-and-watermarking).

## 36. Diagram: watermark with Append output

<img src="assets/structured-streaming-watermark-append-mode.png" alt="Spark watermark diagram: Append mode delays window output until it is finalized" style="background-color:white;max-width:100%;max-height:440px;object-fit:contain;" />

Compare with the previous slide: Append waits until a window can be finalized, then emits its result. This adds output delay but avoids sending repeated revisions of that window.

For window-state cleanup, set the watermark **before** aggregation on the same event-time column and use a supported Append/Update query. Complete mode preserves aggregate history.

Source: [Spark guide — Watermarking](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#handling-late-data-and-watermarking).

## 37. Apply the foundation: payment activity by window

**Context:** `transactions` contains arriving payments with `account_id` and `event_time`. Flag accounts with at least three payments in an aligned five-minute window. `F` denotes `pyspark.sql.functions`.

```python
account_activity = (transactions
    .withWatermark("event_time", "2 minutes")
    .groupBy(F.window("event_time", "5 minutes", "1 minute"), "account_id")
    .agg(F.count("*").alias("payment_count")))

velocity_alerts = account_activity.filter(F.col("payment_count") >= 3)

# Show revised window counts promptly.
velocity_writer = (velocity_alerts.writeStream.format("console")
    .outputMode("update")
    .trigger(processingTime="5 seconds"))
```

**5 minutes:** window width. **1 minute:** window slide. **2 minutes:** lateness allowance. **5 seconds:** execution interval.

This evaluates aligned, overlapping windows, not every possible trailing interval. For one account, counts of **2 → 3 → 4** mean **no alert → qualifying result → revised result**. Use an alert key such as `(account_id, window_start, window_end)` to handle revisions. The threshold illustrates the computation; a business rule needs validation.

## 38. Checkpointing and delivery guarantees

A checkpoint preserves input progress and required state so a compatible query can recover after stopping or failing.

- Use durable storage and one checkpoint location per logical query.
- Reuse that checkpoint for compatible restarts; source/state changes have restrictions.
- Losing progress can cause replay. A retry can repeat an attempted output write.

**Exactly-once results require the whole pipeline to cooperate:** replayable input plus suitable execution and sink behavior. They do not mean each task physically runs only once.

The file sink supports exactly-once output; Kafka output is at least once. Arbitrary database/API writes need idempotency. `foreachBatch` is at least once by default; its implementation must handle retries.

Reference: [Spark guide — Fault Tolerance](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#fault-tolerance-semantics).

## 39. Check your understanding

1. Does `isStreaming=True` prove a query is running?
2. Does a 5-second trigger create 5-second event-time windows?
3. Which trigger fits scheduled incremental ingestion?
4. What happens after `awaitTermination(10)` times out?
5. Why can a high-value filter avoid state while an activity count needs it?
6. Why might Append window output arrive later than Update output?

**Answers:** no, inspect the query handle; no, scheduling and grouping differ; AvailableNow; the query may keep running; current-row comparison versus retained counts; Append waits for finalization.

## 40. Reference map

[Apache Spark 3.5.7 Structured Streaming Programming Guide](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html)

| Topic | Guide section |
|---|---|
| Unbounded table and incremental results | Programming Model / Basic Concepts |
| Readers and schemas | Input Sources |
| Micro-batch scheduling | Triggers |
| Starting, observing and stopping | Managing Streaming Queries |
| Result emission | Output Modes / Output Sinks |
| Event-time grouping | Window Operations / Watermarking |
| Restart behavior | Checkpointing / Recovery Semantics |
| Alternate execution engine | Continuous Processing |

Images © The Apache Software Foundation, reproduced from the linked guide without modification. Local image sources are listed in `assets/SOURCES.md`.